In [10]:
import pandas as pd
import numpy as np
from pathlib import Path


root_folder = Path(r"C:\Users\linjiajun\Desktop\5810data")


def find_file(folder, keyword):
    csv_files = list(folder.glob("*.csv"))

    matched = [
        f for f in csv_files
        if keyword in f.name and "详细" in f.name
    ]

    if len(matched) == 0:
        return None

    if len(matched) > 1:
        matched = sorted(matched, key=lambda x: x.stat().st_mtime, reverse=True)

    return matched[0]


def read_csv_auto(path):
    encodings = ["utf-16", "utf-8-sig", "gbk", "gb18030", "latin1"]
    last_error = None

    for enc in encodings:
        try:
            df = pd.read_csv(
                path,
                encoding=enc,
                sep="\t",
                engine="python"
            )

            df.columns = (
                df.columns
                .str.replace("\ufeff", "", regex=False)
                .str.strip()
            )

            return df

        except Exception as e:
            last_error = e

    raise ValueError(f"无法读取文件：{path}\n最后错误：{last_error}")


def process_one_game(game_folder):
    game_name = game_folder.name

    print("=" * 80)
    print(f"正在处理：{game_name}")

    dau_file = find_file(game_folder, "App Store DAU")
    rev_file = find_file(game_folder, "App Store 收入")

    if dau_file is None:
        print(f"跳过：{game_name}，没有找到 DAU 文件")
        return

    if rev_file is None:
        print(f"跳过：{game_name}，没有找到收入文件")
        return

    print("DAU 文件：", dau_file.name)
    print("收入文件：", rev_file.name)

    dau = read_csv_auto(dau_file)
    rev = read_csv_auto(rev_file)

    required_dau_cols = [
        "App Name",
        "App ID",
        "Date",
        "Country / Region",
        "Platform",
        "Device",
        "DAU"
    ]

    required_rev_cols = [
        "App Name",
        "App ID",
        "Date",
        "Country / Region",
        "Platform",
        "Device",
        "Downloads",
        "Revenue ($)"
    ]

    missing_dau = [col for col in required_dau_cols if col not in dau.columns]
    missing_rev = [col for col in required_rev_cols if col not in rev.columns]

    if missing_dau:
        print(f"跳过：{game_name}，DAU 文件缺少列：{missing_dau}")
        print("实际列名：", list(dau.columns))
        return

    if missing_rev:
        print(f"跳过：{game_name}，收入文件缺少列：{missing_rev}")
        print("实际列名：", list(rev.columns))
        return

    dau["Date"] = pd.to_datetime(dau["Date"], errors="coerce")
    rev["Date"] = pd.to_datetime(rev["Date"], errors="coerce")

    dau["DAU"] = pd.to_numeric(dau["DAU"], errors="coerce")
    rev["Downloads"] = pd.to_numeric(rev["Downloads"], errors="coerce")
    rev["Revenue ($)"] = pd.to_numeric(rev["Revenue ($)"], errors="coerce")

    dau = dau.dropna(subset=["Date"])
    rev = rev.dropna(subset=["Date"])

    group_cols = [
        "App Name",
        "App ID",
        "Date",
        "Country / Region",
        "Platform"
    ]

    dau_daily = (
        dau.groupby(group_cols, as_index=False)
        .agg({"DAU": "sum"})
    )

    rev_daily = (
        rev.groupby(group_cols, as_index=False)
        .agg({
            "Downloads": "sum",
            "Revenue ($)": "sum"
        })
    )

    rev_daily["RPD ($)"] = np.where(
        rev_daily["Downloads"] > 0,
        rev_daily["Revenue ($)"] / rev_daily["Downloads"],
        np.nan
    )

    merged = pd.merge(
        dau_daily,
        rev_daily,
        on=group_cols,
        how="outer"
    )

    merged["ARPDAU ($)"] = np.where(
        merged["DAU"] > 0,
        merged["Revenue ($)"] / merged["DAU"],
        np.nan
    )

    merged = merged.sort_values(
        ["App Name", "Country / Region", "Platform", "Date"]
    )

    merged["Date"] = merged["Date"].dt.strftime("%Y/%m/%d")

    final_cols = [
        "App Name",
        "App ID",
        "Date",
        "Country / Region",
        "Platform",
        "DAU",
        "Downloads",
        "Revenue ($)",
        "RPD ($)",
        "ARPDAU ($)"
    ]

    merged = merged[final_cols]

    output_file = game_folder / f"{game_name}_AppStore_DAU_收入_按日期合并.csv"
    merged.to_csv(output_file, index=False, encoding="utf-8-sig")

    print("处理成功")
    print("输出文件：", output_file)
    print("输出行数：", len(merged))


for game_folder in root_folder.iterdir():
    if game_folder.is_dir():
        try:
            process_one_game(game_folder)
        except Exception as e:
            print("=" * 80)
            print(f"失败：{game_folder.name}")
            print(e)

print("=" * 80)
print("全部游戏处理结束。")

正在处理：原神
DAU 文件： App Store DAU (Sep 25, 2020 - Apr 25, 2026, 3个国家-地区), 详细.csv
收入文件： App Store 收入 (Sep 25, 2020 - Apr 26, 2026, 3个国家-地区), 详细.csv
处理成功
输出文件： C:\Users\linjiajun\Desktop\5810data\原神\原神_AppStore_DAU_收入_按日期合并.csv
输出行数： 8368
正在处理：命运冠位指定
跳过：命运冠位指定，没有找到 DAU 文件
正在处理：少女前线2
DAU 文件： App Store DAU (Dec 20, 2023 - Apr 25, 2026, 3个国家-地区), 详细.csv
收入文件： App Store 收入 (Dec 20, 2023 - Apr 26, 2026, 3个国家-地区), 详细.csv
处理成功
输出文件： C:\Users\linjiajun\Desktop\5810data\少女前线2\少女前线2_AppStore_DAU_收入_按日期合并.csv
输出行数： 1881
正在处理：尘白禁区
DAU 文件： App Store DAU (Jul 18, 2023 - Apr 25, 2026, 3个国家-地区), 详细.csv
收入文件： App Store 收入 (Jul 18, 2023 - Apr 26, 2026, 3个国家-地区), 详细.csv
处理成功
输出文件： C:\Users\linjiajun\Desktop\5810data\尘白禁区\尘白禁区_AppStore_DAU_收入_按日期合并.csv
输出行数： 3041
正在处理：崩坏3
DAU 文件： App Store DAU (Sep 30, 2016 - Apr 25, 2026, 3个国家-地区), 详细.csv
收入文件： App Store 收入 (Sep 30, 2016 - Apr 26, 2026, 3个国家-地区), 详细.csv
处理成功
输出文件： C:\Users\linjiajun\Desktop\5810data\崩坏3\崩坏3_AppStore_DAU_收入_按日期合并.csv
输出行数： 11145
正在处理：崩坏星穹铁道
DAU

In [11]:
import pandas as pd
from pathlib import Path


# =========================
# 1. 总目录
# =========================

root_folder = Path(r"C:\Users\linjiajun\Desktop\5810data")


# =========================
# 2. 总表输出路径
# =========================

output_file = root_folder / "全部游戏_AppStore_DAU_收入_按日期合并总表.csv"


# =========================
# 3. 查找所有已经处理好的文件
# =========================
# 文件名特征：
# 游戏名_AppStore_DAU_收入_按日期合并.csv

processed_files = []

for game_folder in root_folder.iterdir():
    if game_folder.is_dir():
        matched_files = list(game_folder.glob("*_AppStore_DAU_收入_按日期合并.csv"))

        for file in matched_files:
            processed_files.append(file)


print(f"找到已处理文件数量：{len(processed_files)}")

for file in processed_files:
    print(file)


# =========================
# 4. 读取并合并
# =========================

all_data = []

for file in processed_files:
    game_folder_name = file.parent.name

    try:
        df = pd.read_csv(file, encoding="utf-8-sig")

        # 防止列名有空格
        df.columns = df.columns.str.strip()

        # 加一列文件夹游戏名，方便后面核对
        df["Folder Game Name"] = game_folder_name

        all_data.append(df)

        print(f"读取成功：{file.name}，行数：{len(df)}")

    except Exception as e:
        print(f"读取失败：{file}")
        print(e)


# =========================
# 5. 输出总表
# =========================

if all_data:
    merged_all = pd.concat(all_data, ignore_index=True)

    # 日期重新转一下，方便排序
    merged_all["Date"] = pd.to_datetime(merged_all["Date"], errors="coerce")

    merged_all = merged_all.sort_values(
        ["App Name", "Country / Region", "Platform", "Date"]
    )

    merged_all["Date"] = merged_all["Date"].dt.strftime("%Y/%m/%d")

    # 调整列顺序
    final_cols = [
        "Folder Game Name",
        "App Name",
        "App ID",
        "Date",
        "Country / Region",
        "Platform",
        "DAU",
        "Downloads",
        "Revenue ($)",
        "RPD ($)",
        "ARPDAU ($)"
    ]

    # 只保留存在的列，避免个别文件列名异常时报错
    final_cols = [col for col in final_cols if col in merged_all.columns]
    merged_all = merged_all[final_cols]

    merged_all.to_csv(output_file, index=False, encoding="utf-8-sig")

    print("=" * 80)
    print("全部处理好的游戏已汇总完成！")
    print("总表输出：", output_file)
    print("总行数：", len(merged_all))
    print("包含游戏数量：", merged_all["Folder Game Name"].nunique())

else:
    print("没有找到可以合并的处理后文件。")

找到已处理文件数量：17
C:\Users\linjiajun\Desktop\5810data\原神\原神_AppStore_DAU_收入_按日期合并.csv
C:\Users\linjiajun\Desktop\5810data\少女前线2\少女前线2_AppStore_DAU_收入_按日期合并.csv
C:\Users\linjiajun\Desktop\5810data\尘白禁区\尘白禁区_AppStore_DAU_收入_按日期合并.csv
C:\Users\linjiajun\Desktop\5810data\崩坏3\崩坏3_AppStore_DAU_收入_按日期合并.csv
C:\Users\linjiajun\Desktop\5810data\崩坏星穹铁道\崩坏星穹铁道_AppStore_DAU_收入_按日期合并.csv
C:\Users\linjiajun\Desktop\5810data\恋与制作人\恋与制作人_AppStore_DAU_收入_按日期合并.csv
C:\Users\linjiajun\Desktop\5810data\恋与深空\恋与深空_AppStore_DAU_收入_按日期合并.csv
C:\Users\linjiajun\Desktop\5810data\战双帕弥什\战双帕弥什_AppStore_DAU_收入_按日期合并.csv
C:\Users\linjiajun\Desktop\5810data\无期迷途\无期迷途_AppStore_DAU_收入_按日期合并.csv
C:\Users\linjiajun\Desktop\5810data\无限暖暖\无限暖暖_AppStore_DAU_收入_按日期合并.csv
C:\Users\linjiajun\Desktop\5810data\明日方舟\明日方舟_AppStore_DAU_收入_按日期合并.csv
C:\Users\linjiajun\Desktop\5810data\火影忍者\火影忍者_AppStore_DAU_收入_按日期合并.csv
C:\Users\linjiajun\Desktop\5810data\绝区零\绝区零_AppStore_DAU_收入_按日期合并.csv
C:\Users\linjiajun\Desktop\5810data\重返未来1999\重返未来